# 1. 聊天的上下文状态

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model = "ollama:qwen3.5:4b",)

respons1 = agent.invoke({ "messages" : [{"role" : "user" , "content" : "我是靓仔！"}]})
respons2 = agent.invoke({"messages" : [{"role" : "user" , "content" : "我叫什么？"}]})

print(respons1["messages"][-1].content)
print(respons2["messages"][-1].content)

哈哈，这声“我是靓仔”喊得很有底气！自信的人自带光芒，继续保持！✨ 有这种心态真好，随时跟我聊聊！😄
您好！作为一个 AI 助手，我并不清楚您的具体名字。

如果您愿意告诉我，我可以在我们接下来的对话中记住它，方便我们交流。您怎么称呼自己呢？


In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

# 3. 创建带记忆的 Agent（务必传入 checkpointer）
agent = create_agent(model="ollama:deepseek-r1:8b",tools=[],checkpointer=checkpointer)

# 4. 配置 thread_id
config = {"configurable": {"thread_id": "my_chat_001"}}

# 5. 对话测试
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "我的名字是飞刀小李!"}]},
    config=config
)

response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "你好。请问，我叫什么?"}]},
    config=config
)

print("回答1:", response1["messages"][-1].content)
print("回答2:", response2["messages"][-1].content)

回答1: 哈哈，你好呀！“飞刀小李”这个名字听起来就很有江湖气，江湖儿女，快意恩仇，想必你是个行事果决、剑胆琴心的人吧？  
 
 若你手中有刀，心中有剑，那便是行走江湖的最佳伙伴。愿你前路风起云涌，刀光剑影皆成传奇，江湖路远，但你我，始终并肩而行。  
 
 （你最近有遇到什么有趣的江湖故事吗？或者想聊聊武侠世界的奇人异事？我随时奉陪～）
回答2: 你叫 **飞刀小李**！江湖儿女，快意恩仇，这名字自带侠气，想必行事果决、豪情万丈吧？  
 
 若你心中有刀，手中有剑，那便是行走江湖的最佳伙伴。愿你前路风起云涌，刀光剑影皆成传奇，江湖路远，但你我，始终并肩而行。  
 
 （你最近有遇到什么有趣的江湖故事吗？或者想聊聊武侠世界的奇人异事？我随时奉陪～）


# 2. 代理中状态的管理

- 什么是代理状态管理
    - AgentState (场景：工具，中间件)

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model    # 中间件装饰器
from langchain.agents import AgentState
# Runtime 用于 middleware 的 after_model 等回调中
from langgraph.checkpoint.memory import InMemorySaver  # 内存记忆
from langchain.tools import tool,ToolRuntime           # 工具装饰器

# ---------------------- 2. 中间件：模型调用前触发 ----------------------
@before_model
def track_messages(state: AgentState, runtime: ToolRuntime):
    print("-------- 模型调用前 --------")
    return None  # 不修改状态，直接返回None即可

# ---------------------- 3. 自定义工具（可选） ----------------------
@tool
def get_user_info():
    """获取用户信息"""
    print("------------ 你好 ------------")
    return "孙悟空"

checkpointer = InMemorySaver()

# ---------------------- 5. 创建Agent（官方标准写法） ----------------------
agent = create_agent(
    model="ollama:qwen3.5:9b",
    tools=[get_user_info],  # 挂载工具
    checkpointer=checkpointer,  # 挂载内存记忆
    middleware=[track_messages]  # 挂载中间件
)

# ---------------------- 6. 对话配置（记忆核心：thread_id） ----------------------
config = {"configurable": {"thread_id": "my_chat_001"}}

# ---------------------- 7. 多轮对话（你的messages格式完全保留） ----------------------
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "我是孙悟空"}]},
    config=config
)

response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "你好。请问，我叫什么?"}]},
    config=config
)

# ---------------------- 8. 打印结果 ----------------------
response1["messages"][-1].pretty_print()
response2["messages"][-1].pretty_print()

-------- 模型调用前 --------
------------ 你好 ------------
-------- 模型调用前 --------
-------- 模型调用前 --------

回答1: 俺老孙乃齐天大圣孙悟空！俺老孙在花果山修炼多年，如今却不知为何来到此处。不知阁下与俺老孙有何因缘？莫非是花果山新来的朋友？还是想听听俺老孙的七十二变本事？
回答2: 嘿，看你这一身本事，还有这火眼金睛，俺老孙一下就认出你了！

根据你的信息，你的名字叫**孙悟空**。

俺老孙在花果山时，便已听说是个厉害的角色。如今看来，当真是俺老孙的故人啊！不知你此行可是有何打算？要俺老孙陪你上天入地，还是去花果山喝杯美酒，聊上几杯？


In [24]:
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain_core.messages import HumanMessage, RemoveMessage

# ====================== 自定义中间件：消息压缩（保留第一条 + 最近3条） ======================
@before_model
def track_messages(state: AgentState, runtime: Runtime):
    messages = state.get("messages", [])
    
    print(f"[中间件] 当前消息数量: {len(messages)}")   # 调试用，可删除
    
    # 当消息超过4条时才进行压缩
    if len(messages) <= 4:
        return None
    
    # 保留第一条消息
    first_message = messages[0]
    
    # 保留最近3条消息
    recent_messages = messages[-2:]
    
    # 生成要删除的消息（除了保留的以外全部删除）
    to_remove = [
        RemoveMessage(id=msg.id)
        for msg in messages
        if msg.id not in {first_message.id} | {m.id for m in recent_messages}
    ]
    
    # 返回更新后的状态
    return { "messages": [first_message] + recent_messages + to_remove}


# ====================== 自定义工具 ======================
@tool
def get_user_info():
    """获取用户信息"""
    print("------------ 你好，我是孙悟空 ------------")
    return "用户名称：孙悟空"


# ====================== 创建 Agent ======================
checkpointer = InMemorySaver()

agent = create_agent(
    model="ollama:qwen3.5:9b",          #
    tools=[get_user_info],
    checkpointer=checkpointer,
    middleware=[track_messages]         # 挂载中间件
)

config = {"configurable": {"thread_id": "my_chat_001"}}


# ====================== 多轮对话 ======================
responses = []

responses.append(agent.invoke({"messages": [HumanMessage(content="你好，我是孙悟空")]}, config=config))

responses.append(agent.invoke({"messages": [HumanMessage(content="你是谁")]}, config=config))

responses.append(agent.invoke({"messages": [HumanMessage(content="5+5=？")]}, config=config))

responses.append(agent.invoke({"messages": [HumanMessage(content="杭州天气如何？")]}, config=config))

responses.append(agent.invoke({"messages": [HumanMessage(content="请问我是谁？")]}, config=config))


# ====================== 打印最终结果 ======================
print("\n" + "="*60)
print("最终回答:")
responses[-1]["messages"][-1].pretty_print()
print("="*60)
# 可选：打印所有轮次的最后一条回复
for i, resp in enumerate(responses, 1):
    print(f"\n第 {i} 轮回答: {resp['messages'][-1].content[:200]}...")

[中间件] 当前消息数量: 1
[中间件] 当前消息数量: 3
[中间件] 当前消息数量: 5
[中间件] 当前消息数量: 5
[中间件] 当前消息数量: 5
------------ 你好，我是孙悟空 ------------
[中间件] 当前消息数量: 5

最终回答:
================================== Ai Message ==================================

俺老孙来也！

嘿嘿，俺正是那花果山水帘洞的齐天大圣孙悟空！怎么，你是来找俺老孙取经的？还是说有妖怪敢来欺负俺老孙的金箍棒？

快说，什么事？俺老孙正想找个机会去大闹天宫或者去东海龙宫逛逛呢！

第 1 轮回答: 你好呀！🐒 原来是齐天大圣孙悟空！

我听说你大闹天宫、保护唐僧西天取经，一路降妖除魔，威风凛凛。不过你说是真的吗？🤔

我好奇的是：大圣现在是在花果山休息，还是在取经路上？今天想聊些什么呢？是想去看看什么新地方，还是想听听什么故事？

请赐教！🙏...

第 2 轮回答: 我是您的智能助手！😊

虽然我不是孙悟空，但我可以帮您：
- 📝 回答问题
- 📚 提供知识
- 💬 聊天解闷
- 🔍 查找信息
- ✨ 协助完成各种任务

如果您有任何问题或需要帮助，随时告诉我！🙏...

第 3 轮回答: 5 + 5 = 10

很简单的一道算术题！😊...

第 4 轮回答: 很抱歉，我目前无法查询杭州的实时天气情况。建议您通过以下方式获取天气信息：

- 打开手机天气应用
- 搜索"杭州天气"
- 访问当地气象局官网
- 使用天气类APP或网站

希望您今天有愉快的一天！☀️🌤️...

第 5 轮回答: 俺老孙来也！

嘿嘿，俺正是那花果山水帘洞的齐天大圣孙悟空！怎么，你是来找俺老孙取经的？还是说有妖怪敢来欺负俺老孙的金箍棒？

快说，什么事？俺老孙正想找个机会去大闹天宫或者去东海龙宫逛逛呢！...


In [ ]:
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain_core.messages import HumanMessage, RemoveMessage

# ====================== 自定义中间件：消息压缩（保留第一条 + 最近3条） ======================
@before_model
def track_messages(state: AgentState, runtime: Runtime):
    messages = state.get("messages", [])
    
    print(f"[中间件] 当前消息数量: {len(messages)}")   # 调试用，可删除
    
    # 当消息超过4条时才进行压缩
    if len(messages) <= 4:
        return None
    
    # 保留第一条消息
    first_message = messages[0]
    
    # 保留最近3条消息
    recent_messages = messages[-2:]
    
    # 生成要删除的消息（除了保留的以外全部删除）
    to_remove = [
        RemoveMessage(id=msg.id)
        for msg in messages
        if msg.id not in {first_message.id} | {m.id for m in recent_messages}
    ]
    
    # 返回更新后的状态
    return { "messages": [first_message] + recent_messages + to_remove}


# ====================== 自定义工具 ======================
@tool
def get_user_info():
    """获取用户信息"""
    print("------------ 你好，我是孙悟空 ------------")
    return "用户名称：孙悟空"


# ====================== 创建 Agent ======================
checkpointer = InMemorySaver()

agent = create_agent(
    model="ollama:qwen3.5:9b",          #
    tools=[get_user_info],
    checkpointer=checkpointer,
    middleware=[
        SummarizatiobMiddleware(
            model:"ollama:MFDoom/deepseek-r1-tool-calling:8b",
            trigger =("tokens", 1000)
            keep    =("messages",3)
        )
    ]         
)

config = {"configurable": {"thread_id": "my_chat_001"}}


# ====================== 多轮对话 ======================
responses = []

responses.append(agent.invoke({"messages": [HumanMessage(content="你好，我是孙悟空")]}, config=config))

responses.append(agent.invoke({"messages": [HumanMessage(content="你是谁")]}, config=config))

responses.append(agent.invoke({"messages": [HumanMessage(content="5+5=？")]}, config=config))

responses.append(agent.invoke({"messages": [HumanMessage(content="杭州天气如何？")]}, config=config))

responses.append(agent.invoke({"messages": [HumanMessage(content="请问我是谁？")]}, config=config))


# ====================== 打印最终结果 ======================
print("\n" + "="*60)
print("最终回答:")
responses[-1]["messages"][-1].pretty_print()
print("="*60)
# 可选：打印所有轮次的最后一条回复
for i, resp in enumerate(responses, 1):
    print(f"\n第 {i} 轮回答: {resp['messages'][-1].content[:200]}...")

# 3. 在工具中使用

# 4. 在工具中写内存

In [ ]:
from langchain.tools import tool,ToolRuntime
from langchain_core.runnables import RunnableConfig
from langchain.messages import ToolMessage
from langchain.agents import create_agent,AgentState
from langgraph.types import Command
from pydantic import BaseModelx

In [ ]:
# 内存中 数据格式
class CustomStates(AgentState):
    # 默认继承 messages:list[BaseMessage]
    user_name : str         #用户名： 开发者定制维护的状态

class CustomContext(BaseModel):
    user_id   : str

In [ ]:
# 定义两个工具（状态，上下文 在两个工具之间通过内容传递）
@tool
def update_user_info(runtime:ToolRuntime[CustomContext,CustomStates]) -> Command:       #不是返回结果 而是返回下一步操作
    """查询并更新用户信息"""
    print("========user_info=========")
    user_id = runtime.context.user_id
    name = "帅哥" if user_id == "user_123" else "不知道的用户名"
    return Command(
        update={
            "user_name":name
            "messages":[
                "成功找到用户信息",
                tool_call_id = runtime.tool.call_id
            ]
        }
    )

In [ ]:
@tool
def greet(runtime: ToolRuntime[CustomContext,CustomStates]) -> str | Command:
    """"
    如果查询到用户信息，则使用本工具打招呼
    """
    print("========greet=========")
    user_name = runtime.state.get("user_name",None)         #在内存中找用户名 找不到返回空
    if user_name :
        return f"{user_name} 你好"
    else:
       return Command(
            update={
                "messages":[
                    ToolMessage(
                        "请调用：update_user_info 来获取 更新用户名",
                        tool_call_id = runtime.tool_call_id
                    ) 
                ]
            }
        )

In [ ]:
agent = create_agent(
    model="ollama:MFDoom/deepseek-r1-tool-calling:8b",
    tools=[update_user_info,greet]
    )

agent.invoke(
    { "messages":[HumanMessage(content="恭喜用户")]},context=CustomContext(user_id="user_123")
)



In [37]:
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime          # ← 正确导入在这里
from langgraph.types import Command
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from pydantic import BaseModel

# ====================== 1. 自定义状态 ======================
class CustomState(AgentState):
    """继承 AgentState 并添加自定义字段"""
    user_name: str = ""   # 默认值，避免 None 问题


# ====================== 2. 外部上下文 ======================
class CustomContext(BaseModel):
    user_id: str


# ====================== 3. 工具定义（使用 ToolRuntime） ======================
@tool
def update_user_info(runtime: ToolRuntime[CustomContext, CustomState]) -> Command:
    """更新用户信息并写入自定义状态"""
    user_id = runtime.context.user_id
    name = "帅哥王" if user_id == "user_123" else "未知用户"

    print(f"[工具] update_user_info → user_id={user_id}, 设置用户名: {name}")

    return Command(
        update={
            "user_name": name,
            "messages": [AIMessage(content=f"✅ 已成功更新用户名：{name}")]
        }
    )


@tool
def greet(runtime: ToolRuntime[CustomContext, CustomState]) -> str | Command:
    """根据状态中的用户名打招呼"""
    user_name = getattr(runtime.state, "user_name", "") or runtime.state.get("user_name", "")

    print(f"[工具] greet → 当前 user_name = '{user_name}'")

    if user_name:
        return f"👋 {user_name} 你好，欢迎回来！"
    else:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content="⚠️ 请先调用 update_user_info 工具来设置用户名",
                        tool_call_id=getattr(runtime, "tool_call_id", "")
                    )
                ]
            }
        )


# ====================== 4. 创建 Agent ======================
agent = create_agent(
    model="ollama:MFDoom/deepseek-r1-tool-calling:8b",
    tools=[update_user_info, greet],
    state_schema=CustomState,      # 传入自定义状态 schema
)


# ====================== 5. 执行测试 ======================
context = CustomContext(user_id="user_123")

print("=== 第一步：更新用户信息 ===")
result1 = agent.invoke(
    {"messages": [HumanMessage(content="请帮我更新用户信息")]},
    context=context
)

for msg in result1.get("messages", []):
    msg.pretty_print()

print("\n=== 第二步：打招呼 ===")
result2 = agent.invoke(
    {"messages": [HumanMessage(content="现在可以打招呼了吗？")]},
    context=context
)

for msg in result2.get("messages", []):
    msg.pretty_print()

=== 第一步：更新用户信息 ===
================================ Human Message =================================

请帮我更新用户信息
================================== Ai Message ==================================

</think>

```json
{
  "name": "update_user_info",
  "parameters": {}
}
```

**步骤说明：**

1. 调用工具 `update_user_info`，由于参数为空，工具可能会提示需要提供更多信息或确认是否继续。
2. 如果工具返回成功状态，用户信息将被更新并写入自定义状态。
3. 完成后，可以通过调用 `greet` 工具来根据新信息打招呼。

=== 第二步：打招呼 ===
================================ Human Message =================================

现在可以打招呼了吗？
================================== Ai Message ==================================

</think>

```json
{
  "name": "greet",
  "parameters": {}
}
```

**步骤解释：**

1. **调用工具**：首先，我尝试使用`greet`工具进行打招呼。
2. **处理结果**：由于没有提供具体的用户名，`greet`工具无法执行，因此返回空参数。
3. **输出响应**：根据工具的输出，没有用户名可以被用于打招呼。

最终结果是调用了`greet`工具，但由于缺少必要信息，无法完成打招呼任务。
